# D-SAT — Phase 3: Counterfactual Fine-tuning

**Objective:** Take the best Phase 2 checkpoint and fine-tune it on a small set of manually written counterfactual examples to push the model toward genuine causal reasoning rather than pattern matching.

**What is a counterfactual?**  
A counterfactual asks: *what would have happened if the action were different?*  
For example, given the same starting scene (flour in a bowl), if the action is "add salt" the bowl now contains flour and salt. If the action is "add sugar instead", the bowl contains flour and sugar. A model that understands causality should handle both, not pattern-match on "bowl with flour + add something = bowl with stuff".

**Pipeline stages:**
```
1. Environment setup and load Phase 2 checkpoint
2. Define counterfactual triplet set (manually written)
3. Fine-tune on counterfactuals
4. Evaluate: counterfactual differentiation rate + original GED regression check
5. Save final checkpoint
```

> **Runtime:** A100 GPU.  
> **Prerequisites:** Upload `triplets.jsonl` from Phase 1 and `lora_adapter.zip` from Phase 2.

---
## Stage 1 — Setup and load Phase 2 checkpoint

In [ ]:
%%capture
!pip install transformers peft accelerate bitsandbytes sentencepiece protobuf

In [ ]:
import os, json, math, time, zipfile
from pathlib import Path
from typing import Optional

import torch
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig,
)
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset

ROOT           = Path("/content/dsat_phase3")
ROOT.mkdir(exist_ok=True)
TRIPLETS_FILE  = Path("/content/triplets.jsonl")
ADAPTER_ZIP    = Path("/content/lora_adapter.zip")
ADAPTER_DIR    = ROOT / "phase2_adapter"
CF_ADAPTER_DIR = ROOT / "counterfactual_adapter"
CF_ADAPTER_DIR.mkdir(exist_ok=True)

# Unzip Phase 2 adapter
if ADAPTER_ZIP.exists() and not ADAPTER_DIR.exists():
    with zipfile.ZipFile(ADAPTER_ZIP, "r") as z:
        z.extractall(ROOT / "phase2_adapter")
    print("Adapter unzipped.")
elif ADAPTER_DIR.exists():
    print("Adapter already unzipped.")
else:
    raise FileNotFoundError("lora_adapter.zip not found. Please upload it.")

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("HuggingFace token: ")

MODEL_ID = "google/gemma-3-4b-it"

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND'}")
print("Setup complete.")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
)
base_model.config.use_cache = False

print("Loading Phase 2 LoRA adapter...")
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), is_trainable=True)
model.print_trainable_parameters()
print("Phase 2 checkpoint loaded.")

---
## Stage 2 — Counterfactual triplet set

These triplets are written by hand. Each tests whether the model understands that **different actions on the same scene produce different outcomes**.

Counterfactual pairs share the same G_t but differ in `action_text` and `G_t+1`. If the model has learned causal structure rather than surface correlations, it should produce distinct predictions for distinct actions on the same starting state.

10 triplets are used at pilot scale (5 pairs). At full scale, 50-100 examples covering diverse recipe types are recommended.

In [ ]:
# Each pair (a, b) shares the same G_t.
# The action and G_t+1 differ — testing whether the model is action-sensitive.

COUNTERFACTUAL_TRIPLETS = [
    # Pair 1a: add salt to flour
    {
        "segment_id": "cf_001a",
        "action_text": "add salt to the flour",
        "g_t": {
            "objects": [
                {"id": "obj_1", "class": "flour",       "state": {"quantity": "large_amount"}},
                {"id": "obj_2", "class": "mixing_bowl", "state": {"contents": "flour"}},
                {"id": "obj_3", "class": "salt",        "state": {"quantity": "small_amount"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "in",       "object": "obj_2"},
                {"subject": "obj_3", "predicate": "next_to",  "object": "obj_2"}
            ]
        },
        "g_t1": {
            "objects": [
                {"id": "obj_1", "class": "flour",       "state": {"quantity": "large_amount"}},
                {"id": "obj_2", "class": "mixing_bowl", "state": {"contents": "flour_and_salt"}},
                {"id": "obj_3", "class": "salt",        "state": {"quantity": "small_amount"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "in",         "object": "obj_2"},
                {"subject": "obj_3", "predicate": "in",         "object": "obj_2"},
                {"subject": "obj_3", "predicate": "mixed_with", "object": "obj_1"}
            ]
        }
    },
    # Pair 1b: counterfactual — add sugar instead (same G_t, different action)
    {
        "segment_id": "cf_001b",
        "action_text": "add sugar to the flour instead of salt",
        "g_t": {
            "objects": [
                {"id": "obj_1", "class": "flour",       "state": {"quantity": "large_amount"}},
                {"id": "obj_2", "class": "mixing_bowl", "state": {"contents": "flour"}},
                {"id": "obj_3", "class": "salt",        "state": {"quantity": "small_amount"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "in",      "object": "obj_2"},
                {"subject": "obj_3", "predicate": "next_to", "object": "obj_2"}
            ]
        },
        "g_t1": {
            "objects": [
                {"id": "obj_1", "class": "flour",       "state": {"quantity": "large_amount"}},
                {"id": "obj_2", "class": "mixing_bowl", "state": {"contents": "flour_and_sugar"}},
                {"id": "obj_3", "class": "salt",        "state": {"quantity": "small_amount"}},
                {"id": "obj_4", "class": "sugar",       "state": {"quantity": "small_amount"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "in",         "object": "obj_2"},
                {"subject": "obj_4", "predicate": "in",         "object": "obj_2"},
                {"subject": "obj_4", "predicate": "mixed_with", "object": "obj_1"},
                {"subject": "obj_3", "predicate": "next_to",    "object": "obj_2"}
            ]
        }
    },
    # Pair 2a: pour oil into pan
    {
        "segment_id": "cf_002a",
        "action_text": "pour oil into the pan",
        "g_t": {
            "objects": [
                {"id": "obj_1", "class": "frying_pan", "state": {"contents": "empty", "heat": "medium"}},
                {"id": "obj_2", "class": "oil_bottle", "state": {"status": "full"}}
            ],
            "relationships": [
                {"subject": "obj_2", "predicate": "next_to", "object": "obj_1"}
            ]
        },
        "g_t1": {
            "objects": [
                {"id": "obj_1", "class": "frying_pan", "state": {"contents": "oil",   "heat": "medium"}},
                {"id": "obj_2", "class": "oil_bottle", "state": {"status": "partial"}}
            ],
            "relationships": [
                {"subject": "obj_2", "predicate": "poured_into", "object": "obj_1"}
            ]
        }
    },
    # Pair 2b: counterfactual — pour water instead of oil
    {
        "segment_id": "cf_002b",
        "action_text": "pour water into the pan instead of oil",
        "g_t": {
            "objects": [
                {"id": "obj_1", "class": "frying_pan", "state": {"contents": "empty", "heat": "medium"}},
                {"id": "obj_2", "class": "oil_bottle", "state": {"status": "full"}}
            ],
            "relationships": [
                {"subject": "obj_2", "predicate": "next_to", "object": "obj_1"}
            ]
        },
        "g_t1": {
            "objects": [
                {"id": "obj_1", "class": "frying_pan", "state": {"contents": "water",  "heat": "medium"}},
                {"id": "obj_2", "class": "oil_bottle", "state": {"status": "full"}},
                {"id": "obj_3", "class": "water",      "state": {"quantity": "small_amount"}}
            ],
            "relationships": [
                {"subject": "obj_3", "predicate": "in",      "object": "obj_1"},
                {"subject": "obj_2", "predicate": "next_to", "object": "obj_1"}
            ]
        }
    },
    # Pair 3a: chop the onion
    {
        "segment_id": "cf_003a",
        "action_text": "chop the onion",
        "g_t": {
            "objects": [
                {"id": "obj_1", "class": "onion",         "state": {"status": "whole"}},
                {"id": "obj_2", "class": "cutting_board", "state": {}},
                {"id": "obj_3", "class": "knife",         "state": {"status": "clean"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "on",   "object": "obj_2"},
                {"subject": "obj_3", "predicate": "next_to", "object": "obj_1"}
            ]
        },
        "g_t1": {
            "objects": [
                {"id": "obj_1", "class": "onion",         "state": {"status": "chopped"}},
                {"id": "obj_2", "class": "cutting_board", "state": {}},
                {"id": "obj_3", "class": "knife",         "state": {"status": "used"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "on",     "object": "obj_2"},
                {"subject": "obj_3", "predicate": "cutting", "object": "obj_1"}
            ]
        }
    },
    # Pair 3b: counterfactual — slice instead of chop
    {
        "segment_id": "cf_003b",
        "action_text": "slice the onion instead of chopping it",
        "g_t": {
            "objects": [
                {"id": "obj_1", "class": "onion",         "state": {"status": "whole"}},
                {"id": "obj_2", "class": "cutting_board", "state": {}},
                {"id": "obj_3", "class": "knife",         "state": {"status": "clean"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "on",      "object": "obj_2"},
                {"subject": "obj_3", "predicate": "next_to", "object": "obj_1"}
            ]
        },
        "g_t1": {
            "objects": [
                {"id": "obj_1", "class": "onion",         "state": {"status": "sliced"}},
                {"id": "obj_2", "class": "cutting_board", "state": {}},
                {"id": "obj_3", "class": "knife",         "state": {"status": "used"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "on",     "object": "obj_2"},
                {"subject": "obj_3", "predicate": "cutting", "object": "obj_1"}
            ]
        }
    },
    # Pair 4a: add egg to bowl
    {
        "segment_id": "cf_004a",
        "action_text": "crack the egg into the bowl",
        "g_t": {
            "objects": [
                {"id": "obj_1", "class": "bowl",  "state": {"contents": "flour"}},
                {"id": "obj_2", "class": "egg",   "state": {"status": "whole"}}
            ],
            "relationships": [
                {"subject": "obj_2", "predicate": "next_to", "object": "obj_1"}
            ]
        },
        "g_t1": {
            "objects": [
                {"id": "obj_1", "class": "bowl",     "state": {"contents": "flour_and_egg"}},
                {"id": "obj_2", "class": "eggshell", "state": {"status": "cracked"}}
            ],
            "relationships": [
                {"subject": "obj_2", "predicate": "next_to", "object": "obj_1"}
            ]
        }
    },
    # Pair 4b: counterfactual — add milk instead
    {
        "segment_id": "cf_004b",
        "action_text": "pour milk into the bowl instead of cracking an egg",
        "g_t": {
            "objects": [
                {"id": "obj_1", "class": "bowl",  "state": {"contents": "flour"}},
                {"id": "obj_2", "class": "egg",   "state": {"status": "whole"}}
            ],
            "relationships": [
                {"subject": "obj_2", "predicate": "next_to", "object": "obj_1"}
            ]
        },
        "g_t1": {
            "objects": [
                {"id": "obj_1", "class": "bowl",  "state": {"contents": "flour_and_milk"}},
                {"id": "obj_2", "class": "egg",   "state": {"status": "whole"}},
                {"id": "obj_3", "class": "milk",  "state": {"quantity": "small_amount"}}
            ],
            "relationships": [
                {"subject": "obj_3", "predicate": "in",      "object": "obj_1"},
                {"subject": "obj_2", "predicate": "next_to", "object": "obj_1"}
            ]
        }
    },
    # Pair 5a: boil water
    {
        "segment_id": "cf_005a",
        "action_text": "bring the water to a boil",
        "g_t": {
            "objects": [
                {"id": "obj_1", "class": "pot",   "state": {"contents": "water", "heat": "off"}},
                {"id": "obj_2", "class": "stove", "state": {"status": "off"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "on", "object": "obj_2"}
            ]
        },
        "g_t1": {
            "objects": [
                {"id": "obj_1", "class": "pot",   "state": {"contents": "boiling_water", "heat": "high"}},
                {"id": "obj_2", "class": "stove", "state": {"status": "on", "heat": "high"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "on", "object": "obj_2"}
            ]
        }
    },
    # Pair 5b: counterfactual — simmer instead of boil
    {
        "segment_id": "cf_005b",
        "action_text": "set the water to a gentle simmer instead of boiling",
        "g_t": {
            "objects": [
                {"id": "obj_1", "class": "pot",   "state": {"contents": "water", "heat": "off"}},
                {"id": "obj_2", "class": "stove", "state": {"status": "off"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "on", "object": "obj_2"}
            ]
        },
        "g_t1": {
            "objects": [
                {"id": "obj_1", "class": "pot",   "state": {"contents": "simmering_water", "heat": "low"}},
                {"id": "obj_2", "class": "stove", "state": {"status": "on", "heat": "low"}}
            ],
            "relationships": [
                {"subject": "obj_1", "predicate": "on", "object": "obj_2"}
            ]
        }
    }
]

print(f"Counterfactual triplets defined: {len(COUNTERFACTUAL_TRIPLETS)} ({len(COUNTERFACTUAL_TRIPLETS)//2} pairs)")

---
## Stage 3 — Fine-tune on counterfactuals

Training on a small number of epochs (5) with a low learning rate to inject counterfactual signal without causing catastrophic forgetting of Phase 2 knowledge.

In [ ]:
def triplet_to_prompt(triplet: dict) -> dict:
    user_msg = (
        f"Current scene graph:\n{json.dumps(triplet['g_t'], separators=(',', ':'))}\n\n"
        f"Action: {triplet['action_text']}\n\n"
        f"Predict the next scene graph."
    )
    target = json.dumps(triplet["g_t1"], separators=(",", ":"))
    return {"input": user_msg, "target": target}


class TripletDataset(Dataset):
    def __init__(self, triplets, tokenizer, max_length=1024):
        self.samples    = [triplet_to_prompt(t) for t in triplets]
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        messages = [{"role": "user", "content": sample["input"]}]
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        full_text = prompt + sample["target"] + self.tokenizer.eos_token
        enc = self.tokenizer(
            full_text, truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        input_ids = enc["input_ids"][0]
        prompt_ids = self.tokenizer(
            prompt, return_tensors="pt", add_special_tokens=False
        )["input_ids"][0]
        labels = input_ids.clone()
        labels[:len(prompt_ids)] = -100
        return {"input_ids": input_ids, "attention_mask": enc["attention_mask"][0], "labels": labels}


cf_ds = TripletDataset(COUNTERFACTUAL_TRIPLETS, tokenizer)
print(f"Counterfactual dataset: {len(cf_ds)} samples")

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100
)

cf_training_args = TrainingArguments(
    output_dir=str(CF_ADAPTER_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,           # lower LR to reduce catastrophic forgetting risk
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=2,
    save_strategy="no",
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

cf_trainer = Trainer(
    model=model,
    args=cf_training_args,
    train_dataset=cf_ds,
    data_collator=data_collator,
)

print("Starting counterfactual fine-tuning...")
cf_trainer.train()

---
## Stage 4 — Evaluate

Two evaluations are run:

1. **Counterfactual differentiation rate** — for each pair, do the model's predictions for action A and action B actually differ? A causally-aware model should differentiate all pairs.

2. **GED regression check** — re-run the original Phase 2 val set and confirm the mean GED has not degraded significantly (target: within 0.05 of the Phase 2 baseline of 0.325). A large increase indicates catastrophic forgetting.

In [ ]:
def generate_prediction(model, tokenizer, triplet: dict, max_new_tokens=512) -> Optional[dict]:
    prompt   = triplet_to_prompt(triplet)
    messages = [{"role": "user", "content": prompt["input"]}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    try:
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        return None

In [ ]:
# Test 1: counterfactual pair differentiation
model.eval()
print("=" * 60)
print("COUNTERFACTUAL PAIR DIFFERENTIATION TEST")
print("=" * 60)

pairs_tested       = 0
pairs_differentiated = 0

for i in range(0, len(COUNTERFACTUAL_TRIPLETS) - 1, 2):
    t_a = COUNTERFACTUAL_TRIPLETS[i]
    t_b = COUNTERFACTUAL_TRIPLETS[i + 1]

    pred_a = generate_prediction(model, tokenizer, t_a)
    pred_b = generate_prediction(model, tokenizer, t_b)

    if pred_a is None or pred_b is None:
        print(f"Pair {i//2 + 1}: parse failed")
        continue

    different = json.dumps(pred_a, sort_keys=True) != json.dumps(pred_b, sort_keys=True)
    pairs_tested += 1
    if different:
        pairs_differentiated += 1

    print(f"\nPair {i//2 + 1}:")
    print(f"  Action A: {t_a['action_text']}")
    print(f"  Action B: {t_b['action_text']}")
    print(f"  Predictions differ: {different}")
    if different:
        objs_a = {o['class'] for o in pred_a.get('objects', [])}
        objs_b = {o['class'] for o in pred_b.get('objects', [])}
        print(f"  Pred A objects: {objs_a}")
        print(f"  Pred B objects: {objs_b}")

print(f"\nDifferentiation rate: {pairs_differentiated}/{pairs_tested} pairs")
print("A causally-aware model should differentiate all pairs.")

In [ ]:
# Test 2: GED regression check against Phase 2 baseline
import networkx as nx
import random

def graph_to_nx(graph):
    G = nx.Graph()
    for obj in graph.get("objects", []):
        state_str = "_".join(f"{k}:{v}" for k, v in obj.get("state", {}).items())
        G.add_node(obj["id"], label=f"{obj['class']}|{state_str}")
    for rel in graph.get("relationships", []):
        G.add_edge(rel["subject"], rel["object"], label=rel["predicate"])
    return G

def compute_ged(g_pred, g_true):
    try:
        G_pred = graph_to_nx(g_pred)
        G_true = graph_to_nx(g_true)
        ged = nx.graph_edit_distance(G_pred, G_true, timeout=2.0)
        max_size = max(
            G_pred.number_of_nodes() + G_pred.number_of_edges(),
            G_true.number_of_nodes() + G_true.number_of_edges(), 1
        )
        return ged / max_size
    except Exception:
        return 1.0

# Reconstruct the same val split used in Phase 2
original_triplets = []
with open(TRIPLETS_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            original_triplets.append(json.loads(line))

random.seed(42)
random.shuffle(original_triplets)
val_triplets = original_triplets[int(0.8 * len(original_triplets)):]

print(f"Re-evaluating on {len(val_triplets)} original val triplets...")
ged_scores = []
parse_ok   = 0

for triplet in tqdm(val_triplets):
    pred = generate_prediction(model, tokenizer, triplet)
    if pred is None:
        ged_scores.append(1.0)
        continue
    parse_ok += 1
    ged_scores.append(compute_ged(pred, triplet["g_t1"]))

mean_ged = sum(ged_scores) / len(ged_scores)
print(f"\nPost-CF fine-tuning results on original val set:")
print(f"  JSON parse rate : {parse_ok/len(val_triplets):.0%}")
print(f"  Mean norm. GED  : {mean_ged:.3f}")
print(f"  Phase 2 baseline: 0.325  (delta should be < 0.05 to confirm no catastrophic forgetting)")

---
## Stage 5 — Save final checkpoint

In [ ]:
final_adapter_path = ROOT / "final_adapter"
model.save_pretrained(str(final_adapter_path))
tokenizer.save_pretrained(str(final_adapter_path))

size_mb = sum(
    f.stat().st_size for f in final_adapter_path.rglob("*") if f.is_file()
) / 1e6
print(f"Final adapter saved: {final_adapter_path} ({size_mb:.1f} MB)")

import shutil
from google.colab import files

shutil.make_archive(str(ROOT / "final_adapter"), "zip", str(final_adapter_path))
files.download(str(ROOT / "final_adapter.zip"))
print("Phase 3 complete. final_adapter.zip is the model checkpoint for Phase 4.")

---
## Expected results at pilot scale

**Counterfactual differentiation rate:** With 10 examples and 5 epochs, expect 3-5 out of 5 pairs to be correctly differentiated. A 0/5 rate indicates the model is ignoring the action entirely, which would suggest Phase 2 training did not learn action-conditioned prediction.

**GED vs Phase 2 baseline:** Should stay within ~0.05 of 0.325. A jump above 0.45 indicates catastrophic forgetting — reduce the learning rate to 1e-5 and re-run Phase 3.

**At full scale (50+ counterfactual examples):**
- Differentiation rate should reach 80%+
- GED should improve slightly as counterfactual training reinforces action-conditioned reasoning